# 07 - Rollback e recuperação operacional

## Objetivo

Simular recuperação de uma tabela após uma carga errada.

## Valor para a PRODEMGE

Em caso de erro operacional, a equipe pode restaurar a tabela para um snapshot anterior.

## Como usar

1. Substitua os placeholders (`<datahub_link>`, `<usuário>` e `<senha>`) no primeiro bloco de código.
2. Execute a célula de configuração Livy.
3. Execute as células da demo.
4. No final, encerre a sessão Livy.

> Este notebook não executa Spark localmente. Todo código Spark SQL/PySpark é submetido ao Livy3 via REST API.

In [ ]:
import requests
import time
import json
import urllib3

urllib3.disable_warnings()

# =====================================================================
# CONFIGURAÇÕES DO LIVY3 - AJUSTE ESTES VALORES
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"

# =====================================================================
# CONFIGURAÇÕES DE SESSÃO SPARK
# =====================================================================

# Se o catálogo iceberg_prod já estiver configurado no cluster,
# você pode remover as configs spark.sql.catalog.* abaixo.
SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Demo_Livy3",
    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",
    "spark.driver.memory": "2g",

    # Catálogo Iceberg. Ajuste conforme o padrão do seu ambiente.
    "spark.sql.catalog.iceberg_prod": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.iceberg_prod.type": "hive",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE LIVY3
# =====================================================================

http = requests.Session()
http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {"Content-Type": "application/json"}


def pretty_json(obj):
    """Imprime JSON formatado para facilitar troubleshooting."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


def create_livy_session(kind="pyspark", conf=None, timeout_seconds=600):
    """
    Cria uma sessão Livy interativa.

    kind="pyspark" permite enviar código PySpark e Spark SQL usando spark.sql(...).
    """
    payload = {
        "kind": kind,
        "conf": conf or SESSION_CONF
    }

    response = http.post(
        f"{LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload)
    )
    response.raise_for_status()

    session_id = response.json()["id"]
    print(f"Sessão Livy criada: {session_id}")

    start = time.time()

    while True:
        response = http.get(f"{LIVY_URL}/sessions/{session_id}")
        response.raise_for_status()

        payload = response.json()
        state = payload.get("state")
        print(f"Estado da sessão: {state}")

        if state == "idle":
            print("Sessão Livy pronta para receber statements.")
            return session_id

        if state in ["dead", "error", "killed"]:
            pretty_json(payload)
            raise RuntimeError(f"Falha ao criar sessão Livy. Estado: {state}")

        if time.time() - start > timeout_seconds:
            raise TimeoutError("Timeout aguardando sessão Livy ficar idle.")

        time.sleep(5)


def submit_statement(session_id, code, kind="pyspark", timeout_seconds=900):
    """
    Submete um statement para uma sessão Livy existente.

    O código enviado deve ser uma string Python válida.
    Para SQL, use spark.sql(\"\"\" ... \"\"\").show().
    """
    payload = {
        "code": code,
        "kind": kind
    }

    response = http.post(
        f"{LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload)
    )
    response.raise_for_status()

    statement_id = response.json()["id"]
    print(f"Statement submetido: {statement_id}")

    start = time.time()

    while True:
        response = http.get(
            f"{LIVY_URL}/sessions/{session_id}/statements/{statement_id}"
        )
        response.raise_for_status()

        result = response.json()
        state = result.get("state")
        print(f"Estado do statement: {state}")

        if state == "available":
            output = result.get("output", {})
            pretty_json(output)
            return output

        if state in ["error", "cancelling", "cancelled"]:
            pretty_json(result)
            raise RuntimeError(f"Statement falhou. Estado: {state}")

        if time.time() - start > timeout_seconds:
            raise TimeoutError("Timeout aguardando statement finalizar.")

        time.sleep(3)


def close_livy_session(session_id):
    """Encerra a sessão Livy para liberar recursos no cluster."""
    response = http.delete(f"{LIVY_URL}/sessions/{session_id}")
    if response.status_code in [200, 202, 204]:
        print(f"Sessão Livy encerrada: {session_id}")
    else:
        print(f"Não foi possível encerrar a sessão {session_id}.")
        print(response.text)


# Cria uma sessão Livy para este notebook.
livy_session_id = create_livy_session()

In [ ]:
# Normaliza o username para usar no nome da tabela sem caracteres inválidos.
table_suffix = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in USERNAME).strip("_")
table_name = f"beneficios_{table_suffix}"
particionado_table = f"beneficios_particionados_{table_suffix}"
staging_table = f"beneficios_staging_{table_suffix}"
csv_table = f"beneficios_csv_{table_suffix}"

spark_code = f"""
# =====================================================================
# ROLLBACK DE SNAPSHOT
# =====================================================================

# Simula uma carga incorreta.
spark.sql(\"\"\"
INSERT INTO governo_mg.{table_name} VALUES
(999999, 'Carga Indevida', 999999.99, DATE '2025-12-31', 'ERRO')
\"\"\")

print("Tabela após carga indevida:")
spark.sql(\"\"\"
SELECT *
FROM governo_mg.{table_name}
WHERE id_cidadao = 999999
\"\"\").show(truncate=False)

# Lista snapshots.
snapshots_df = spark.sql(\"\"\"
SELECT snapshot_id, committed_at, operation
FROM governo_mg.{table_name}.snapshots
ORDER BY committed_at DESC
\"\"\")

snapshots_df.show(truncate=False)
snapshots = snapshots_df.collect()

# O snapshot anterior ao mais recente deve representar o estado antes da carga indevida.
if len(snapshots) > 1:
    rollback_snapshot = snapshots[1]["snapshot_id"]
    print(f"Snapshot de rollback: {rollback_snapshot}")

    # Chamada de procedure Iceberg.
    # A procedure depende do catálogo estar configurado como SparkCatalog.
    spark.sql(f\"\"\"
    CALL system.rollback_to_snapshot(
        table => 'governo_mg.{table_name}',
        snapshot_id => {rollback_snapshot}
    )
    \"\"\")

    print("Tabela após rollback:")
    spark.sql(\"\"\"
    SELECT *
    FROM governo_mg.{table_name}
    WHERE id_cidadao = 999999
    \"\"\").show(truncate=False)

    print("Rollback executado com sucesso.")
else:
    print("Não há snapshots suficientes para demonstrar rollback.")
"""

submit_statement(livy_session_id, spark_code)

In [ ]:
# Encerre a sessão ao final do notebook.
close_livy_session(livy_session_id)